In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import mlflow
import mlflow.sklearn
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.metrics import roc_auc_score, confusion_matrix, ConfusionMatrixDisplay
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
import lightgbm as lgb
import warnings
warnings.filterwarnings('ignore')

In [ ]:
# configuration MLFlow
mlflow.set_tracking_uri('sqlite:///mlflow.db')
mlflow.set_experiment('scoring_credit')

In [ ]:
# chargement du dataset préparé et split train/test
df = pd.read_csv('../data/application_train_prepared.csv')

# séparation features / cible
X = df.drop(columns=['TARGET', 'SK_ID_CURR'])
y = df['TARGET']

print(f"Shape X : {X.shape}")
print(f"Shape y : {y.shape}")
print(f"\nDistribution de y :\n{y.value_counts(normalize=True).round(3)}")

# split train/test stratifié
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,        
    random_state=42,      
    stratify=y            
)

print(f"\nShape X_train : {X_train.shape}")
print(f"Shape X_test  : {X_test.shape}")
print(f"\nDistribution y_train : {y_train.value_counts(normalize=True).round(3).to_dict()}")
print(f"Distribution y_test  : {y_test.value_counts(normalize=True).round(3).to_dict()}")

In [ ]:
# score métier

def score_metier(y_true, y_pred, cout_fn=10, cout_fp=1):
    """
    Calcule le coût métier normalisé d'un modèle de classification.
    
    Paramètres :
    - y_true   : valeurs réelles (0 ou 1)
    - y_pred   : valeurs prédites (0 ou 1)
    - cout_fn  : coût d'un faux négatif (défaut par défaut = 10)
    - cout_fp  : coût d'un faux positif (défaut par défaut = 1)
    
    Retourne :
    - coût normalisé (entre 0 et 1, plus c'est bas mieux c'est)
    """
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
    
    cout_total = (fn * cout_fn) + (fp * cout_fp)
    
    # normalisation : on divise par le pire cas possible
    pire_cas = y_true.sum() * cout_fn
    
    return cout_total / pire_cas


def optimiser_seuil(y_true, y_proba, cout_fn=10, cout_fp=1):
    """
    Trouve le seuil optimal qui minimise le score métier.
    
    Paramètres :
    - y_true  : valeurs réelles
    - y_proba : probabilités prédites pour la classe 1
    
    Retourne :
    - seuil optimal
    - score métier au seuil optimal
    """
    seuils = np.arange(0.01, 0.99, 0.01)
    scores = []
    
    for seuil in seuils:
        y_pred = (y_proba >= seuil).astype(int)
        scores.append(score_metier(y_true, y_pred, cout_fn, cout_fp))
    
    idx_optimal = np.argmin(scores)
    return seuils[idx_optimal], scores[idx_optimal]


In [ ]:
# scorer metier pour la cross-validation
# GridSearchCV maximise le score renvoye -> on renvoie l'oppose du cout
# (un cout, ca se minimise). Pour chaque combinaison d'hyperparametres,
# on optimise d'abord le seuil, puis on evalue le cout metier a ce seuil.
# signature (estimator, X, y) : robuste, independante de la version de sklearn
# (pas de make_scorer / needs_proba / response_method a gerer).

def score_metier_cv(estimator, X, y):
    y_proba = estimator.predict_proba(X)[:, 1]
    _, cout = optimiser_seuil(y, y_proba, cout_fn=10, cout_fp=1)
    return -cout


In [ ]:
# baseline - DummyClassifier
with mlflow.start_run(run_name="baseline_dummy"):
    
    # modèle qui prédit toujours la classe majoritaire
    dummy = DummyClassifier(strategy='most_frequent', random_state=42)
    dummy.fit(X_train, y_train)
    
    # prédictions
    y_pred_dummy = dummy.predict(X_test)
    y_proba_dummy = dummy.predict_proba(X_test)[:, 1]
    
    # métriques
    auc_dummy = roc_auc_score(y_test, y_proba_dummy)
    seuil_dummy, cout_dummy = optimiser_seuil(y_test, y_proba_dummy)
    
    # log MLFlow
    mlflow.log_metric("auc", auc_dummy)
    mlflow.log_metric("cout_metier", cout_dummy)
    mlflow.log_metric("seuil_optimal", seuil_dummy)
    mlflow.log_param("model", "DummyClassifier")
    mlflow.log_param("strategy", "most_frequent")
    
    print(f"AUC          : {auc_dummy:.4f}")
    print(f"Coût métier  : {cout_dummy:.4f}")
    print(f"Seuil optimal: {seuil_dummy:.2f}")

In [ ]:
# modèle 1 - Régression Logistique
from sklearn.linear_model import LogisticRegression

with mlflow.start_run(run_name="logistic_regression"):
    
    lr = LogisticRegression(
        max_iter=1000,
        class_weight='balanced', 
        random_state=42,
        n_jobs=-1
    )
    lr.fit(X_train, y_train)
    
    # prédictions
    y_proba_lr = lr.predict_proba(X_test)[:, 1]
    
    # métriques
    auc_lr = roc_auc_score(y_test, y_proba_lr)
    seuil_lr, cout_lr = optimiser_seuil(y_test, y_proba_lr)
    
    # log MLFlow
    mlflow.log_metric("auc", auc_lr)
    mlflow.log_metric("cout_metier", cout_lr)
    mlflow.log_metric("seuil_optimal", seuil_lr)
    mlflow.log_param("model", "LogisticRegression")
    mlflow.log_param("class_weight", "balanced")
    mlflow.log_param("max_iter", 1000)
    mlflow.sklearn.log_model(lr, "model")
    
    print(f"=== Régression Logistique ===")
    print(f"AUC          : {auc_lr:.4f}")
    print(f"Coût métier  : {cout_lr:.4f}")
    print(f"Seuil optimal: {seuil_lr:.2f}")

In [ ]:
# modèle 2 - LightGBM
with mlflow.start_run(run_name="lightgbm_baseline"):
    
    lgbm = lgb.LGBMClassifier(
        n_estimators=500,
        learning_rate=0.05,
        num_leaves=31,
        class_weight='balanced',
        random_state=42,
        n_jobs=-1,
        verbose=-1      
    )
    lgbm.fit(
        X_train, y_train,
        eval_set=[(X_test, y_test)], 
    )
    
    # prédictions
    y_proba_lgbm = lgbm.predict_proba(X_test)[:, 1]
    
    # métriques
    auc_lgbm = roc_auc_score(y_test, y_proba_lgbm)
    seuil_lgbm, cout_lgbm = optimiser_seuil(y_test, y_proba_lgbm)
    
    # log MLFlow
    mlflow.log_metric("auc", auc_lgbm)
    mlflow.log_metric("cout_metier", cout_lgbm)
    mlflow.log_metric("seuil_optimal", seuil_lgbm)
    mlflow.log_param("model", "LightGBM")
    mlflow.log_param("n_estimators", 500)
    mlflow.log_param("learning_rate", 0.05)
    mlflow.log_param("num_leaves", 31)
    mlflow.log_param("class_weight", "balanced")
    mlflow.sklearn.log_model(lgbm, "model")
    
    print(f"=== LightGBM baseline ===")
    print(f"AUC          : {auc_lgbm:.4f}")
    print(f"Coût métier  : {cout_lgbm:.4f}")
    print(f"Seuil optimal: {seuil_lgbm:.2f}")

In [ ]:
# optimisation LightGBM - GridSearchCV
from sklearn.model_selection import GridSearchCV

param_grid = {
    'n_estimators'  : [300, 500],
    'learning_rate' : [0.05, 0.1],
    'num_leaves'    : [31, 63],
    'min_child_samples': [20, 50],
}

# modèle de base
lgbm_base = lgb.LGBMClassifier(
    class_weight='balanced',
    random_state=42,
    n_jobs=-1,
    verbose=-1
)

# cross-validation stratifiée 
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

grid_search = GridSearchCV(
    estimator=lgbm_base,
    param_grid=param_grid,
    cv=cv,
    scoring=score_metier_cv,   # score metier au lieu de roc_auc
    n_jobs=-1 
)

grid_search.fit(X_train, y_train)

print(f"Meilleurs paramètres : {grid_search.best_params_}")
# best_score_ est negatif (cout negative par le scorer) -> on re-negative
print(f"Meilleur cout metier (CV): {-grid_search.best_score_:.4f}")

In [ ]:
# modèle final - LightGBM optimisé
with mlflow.start_run(run_name="lightgbm_optimized"):
    
    best_params = grid_search.best_params_
    
    lgbm_final = lgb.LGBMClassifier(
        **best_params,
        class_weight='balanced',
        random_state=42,
        n_jobs=-1,
        verbose=-1
    )
    lgbm_final.fit(X_train, y_train)
    
    # prédictions
    y_proba_final = lgbm_final.predict_proba(X_test)[:, 1]
    
    # métriques
    auc_final = roc_auc_score(y_test, y_proba_final)
    seuil_final, cout_final = optimiser_seuil(y_test, y_proba_final)
    y_pred_final = (y_proba_final >= seuil_final).astype(int)
    
    # log MLFlow
    mlflow.log_params(best_params)
    mlflow.log_param("model", "LightGBM_optimized")
    mlflow.log_param("class_weight", "balanced")
    mlflow.log_metric("auc", auc_final)
    mlflow.log_metric("cout_metier", cout_final)
    mlflow.log_metric("seuil_optimal", seuil_final)
    mlflow.log_metric("cout_metier_cv", -grid_search.best_score_)
    mlflow.sklearn.log_model(
        lgbm_final,
        "model",
        registered_model_name="scoring_credit_lgbm"
    )
    
    print(f"=== LightGBM optimisé ===")
    print(f"AUC test         : {auc_final:.4f}")
    print(f"Coût métier (CV) : {-grid_search.best_score_:.4f}")
    print(f"Coût métier      : {cout_final:.4f}")
    print(f"Seuil optimal    : {seuil_final:.2f}")
    
    # matrice de confusion
    fig, ax = plt.subplots(figsize=(6, 4))
    ConfusionMatrixDisplay.from_predictions(y_test, y_pred_final, ax=ax)
    ax.set_title(f"Matrice de confusion - seuil={seuil_final:.2f}")
    plt.tight_layout()
    mlflow.log_figure(fig, "confusion_matrix.png")
    plt.show()

In [ ]:
# feature Importance Globale
import shap

# LightGBM a sa propre feature importance intégrée
feature_importance = pd.DataFrame({
    'feature'   : X_train.columns,
    'importance': lgbm_final.feature_importances_
}).sort_values('importance', ascending=False)

# top 20 features
fig, ax = plt.subplots(figsize=(10, 8))
sns.barplot(
    data=feature_importance.head(20),
    x='importance',
    y='feature',
    palette='viridis',
    ax=ax
)
ax.set_title("Top 20 - Feature Importance Globale (LightGBM)")
ax.set_xlabel("Importance")
ax.set_ylabel("")
plt.tight_layout()
plt.savefig('../models/feature_importance_globale.png')
plt.show()

print(f"\nTop 10 features :")
print(feature_importance.head(10).to_string(index=False))

In [ ]:
# feature Importance Locale - SHAP
sample_size = 1000
X_sample = X_test.iloc[:sample_size]

explainer = shap.TreeExplainer(lgbm_final)
shap_explanation = explainer(X_sample)

# classif binaire : SHAP peut renvoyer une dimension par classe.
# on isole la classe positive (defaut = 1) pour rester en 2D.
if shap_explanation.values.ndim == 3:
    shap_explanation = shap_explanation[:, :, 1]

print(f"Shape shap_explanation.values : {shap_explanation.values.shape}")

In [ ]:
shap_vals = shap_explanation.values
print(f"Shape shap_values : {shap_vals.shape}")

# summary plot global --
plt.figure()
shap.summary_plot(
    shap_vals,
    X_sample,
    max_display=20,
    show=False
)
plt.title("SHAP - Feature Importance Globale")
plt.tight_layout()
plt.savefig('../models/shap_summary_global.png')
plt.show()

In [ ]:
# feature Importance Locale - SHAP (un client)

# on prend le premier client de l'échantillon
client_idx = 0
client = X_sample.iloc[[client_idx]]
proba_client = lgbm_final.predict_proba(client)[0][1]
print(f"Probabilité de défaut du client {client_idx} : {proba_client:.4f}")
print(f"Décision (seuil={seuil_final}) : {'REFUSÉ' if proba_client >= seuil_final else 'ACCORDÉ'}")

# waterfall plot 
plt.figure()
shap.waterfall_plot(
    shap.Explanation(
        values=shap_explanation.values[client_idx],
        base_values=shap_explanation.base_values[client_idx],
        data=X_sample.iloc[client_idx],
        feature_names=X_sample.columns.tolist()
    ),
    max_display=15,
    show=False
)
plt.title(f"SHAP - Explication locale client {client_idx}")
plt.tight_layout()
plt.savefig('../models/shap_local_client.png')
plt.show()

In [ ]:
import joblib

joblib.dump(lgbm_final, '../models/lgbm_final.pkl')
joblib.dump(seuil_final, '../models/seuil_optimal.pkl')

print(f"Seuil optimal sauvegardé : {seuil_final}")

In [ ]:
import json

client_exemple = X_test.iloc[0].to_dict()

print(json.dumps(client_exemple, indent=2)[:500])

In [ ]:
import requests

response = requests.post(
    'http://127.0.0.1:5001/predict',
    json=client_exemple
)

print(f"Status HTTP : {response.status_code}")
print(f"Réponse : {response.json()}")

In [ ]:
API_URL = "https://rmercierwork-scoring-credit-p7.hf.space"

response_health = requests.get(f"{API_URL}/health")
print(f"Health check : {response_health.json()}")

response_predict = requests.post(f"{API_URL}/predict", json=client_exemple)
print(f"Prédiction : {response_predict.json()}")